# Pig Runner + r2dreamer, on Colab's free GPU

Trains DreamerV3 (via [r2dreamer](https://github.com/NM512/r2dreamer)) on the
Pig Runner environment, then renders a rollout of the trained agent to a GIF
and shows it right here — no terminal, no window, nothing local required.

**Before running:** Runtime -> Change runtime type -> GPU.

Replace `PIG_RUNNER_REPO` below with this repo's clone URL.

In [ ]:
PIG_RUNNER_REPO = "https://github.com/Sweeyya/pig-runner.git"
R2DREAMER_REPO = "https://github.com/NM512/r2dreamer.git"
LOGDIR = "/content/pigrunner_run"
TRAIN_STEPS = 5e4   # small demo run -- bump way up for anything close to the
                    # ~17-19 score the scripted expert gets

In [ ]:
!git clone -q $PIG_RUNNER_REPO /content/pig-runner
!git clone -q $R2DREAMER_REPO /content/r2dreamer
%cd /content/r2dreamer
!pip install -q -r requirements.txt
!pip install -q -r /content/pig-runner/requirements.txt

## Wire the environment in

Copies the pure logic files into r2dreamer's `envs/`, drops in the config,
and patches `make_env()` with the `pigrunner` branch -- the same three steps
from the main README, done here instead of by hand.

In [ ]:
init_path = envs_dir / "__init__.py"
init_src = init_path.read_text()

branch_lines = (src / "integration/make_env_branch.py").read_text().splitlines()
branch_code = "\n".join(l for l in branch_lines if not l.strip().startswith("#"))

marker = '    else:\n        raise NotImplementedError(suite)'
assert marker in init_src, "envs/__init__.py layout changed upstream -- patch make_env() by hand"
init_src = init_src.replace(marker, branch_code.rstrip("\n") + "\n" + marker)
init_path.write_text(init_src)
print("Patched envs/__init__.py with the pigrunner branch.")


## Train

`time_limit: 1000` in `pigrunner.yaml` is r2dreamer's own `TimeLimit`
wrapper doing the 1000-step cap -- this env only ever reports real death.

In [ ]:
!python train.py env=pigrunner logdir=$LOGDIR env.steps=$TRAIN_STEPS device=cuda:0

## Watch the trained agent

Loads `latest.pt`, rolls out one episode with the environment's own headless
renderer (`render_rgb` -- the same one `play.py` uses locally, no display
server needed), and writes a GIF.

In [ ]:
import sys
sys.path.append("/content/r2dreamer")
import torch
import hydra
from dreamer import Dreamer
from envs import wrappers
from envs.pigrunner import PigRunnerEnv

device = "cuda:0" if torch.cuda.is_available() else "cpu"
raw_env = PigRunnerEnv(task="v0", seed=0)
env = wrappers.OneHotAction(raw_env)

# Rebuild the exact config train.py used (env=pigrunner), the same way its
# own @hydra.main decorator does, so the model architecture matches the
# checkpoint's weights.
with hydra.initialize(config_path="configs", version_base=None):
    config = hydra.compose(config_name="configs", overrides=["env=pigrunner"])

agent = Dreamer(config.model, env.observation_space, env.action_space)
ckpt = torch.load(f"{LOGDIR}/latest.pt", map_location=device)
agent.load_state_dict(ckpt["agent_state_dict"])
agent.to(device).eval()
print("Loaded", f"{LOGDIR}/latest.pt")


In [ ]:
obs = env.reset()
state = agent.get_initial_state(1)
frames, score = [], 0
for _ in range(1000):
    batched = {
        k: torch.as_tensor(v, device=device, dtype=torch.float32)[None]
        if k == "state" else torch.as_tensor([v], device=device)
        for k, v in obs.items()
    }
    action, state = agent.act(batched, state, eval=True)
    obs, reward, done, info = env.step(action[0].cpu().numpy())
    score += reward
    frames.append(raw_env.render())
    if done:
        break
print(f"episode score: {score}, length: {len(frames)}")


In [ ]:
import imageio
from IPython.display import Image, display

imageio.mimsave("/content/rollout.gif", frames, fps=50, loop=0)
display(Image(filename="/content/rollout.gif"))

## Training curve

r2dreamer logs scalars with `tools.Logger` under `logdir` (TensorBoard
event files). Point TensorBoard at it directly rather than re-parsing logs
by hand:

In [ ]:
%load_ext tensorboard
%tensorboard --logdir $LOGDIR